In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer
dataset = load_dataset("jeggers/CoT-Collection")

dataset

DatasetDict({
    train: Dataset({
        features: ['final_input', 'final_target', 'question_id', 'Canary String', 'dataset'],
        num_rows: 62000
    })
    finetune: Dataset({
        features: ['final_input', 'final_target', 'question_id', 'Canary String', 'dataset'],
        num_rows: 16000
    })
    test_in_dist: Dataset({
        features: ['final_input', 'final_target', 'question_id', 'Canary String', 'dataset'],
        num_rows: 2000
    })
    test_out_dist: Dataset({
        features: ['final_input', 'final_target', 'question_id', 'Canary String', 'dataset'],
        num_rows: 2731
    })
})

In [4]:
df = dataset["train"].to_pandas()
example = df[df["question_id"] == "DMath: train#3242"]
example

,final_input,final_target,question_id,Canary String,dataset
43268,Subtracting 17 from a number gives 55. If Yoon...,8,DMath: train#3242,None,DMath


In [22]:
tokenizer = AutoTokenizer.from_pretrained("jeggers/gemma-2-2b-cot-finetuned")

text = r"<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><bos>Subtracting 17 from a number gives 55. If Yoongi wants to divide the number by 9, how much does Yoongi get? BOT: 1. Let the number be \( x \). 2. According to the problem, subtracting 17 from \( x \) results in 55: \[ x - 17 = 55 \] 3. Solve for \( x \): \[ x = 55 + 17 = 72 \] 4. Now, Yoongi wants to divide this number by 9. \[ \text{Quotient} = x / 9 \] 5. Substituting the calculated value of \( x \): \[ \text{Quotient} = 72 / 9 = 8 \] 6. The division result is 8. ANSWER: 8<eos>"
print(text)

tokens = tokenizer.encode(text, add_special_tokens=False)
print(tokens)

decoded = tokenizer.decode(tokens, skip_special_tokens=True)
print(decoded)

<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><bos>Subtracting 17 from a number gives 55. If Yoongi wants to divide the number by 9, how much does Yoongi get? BOT: 1. Let the number be \( x \). 2. According to the problem, subtracting 17 from \( x \) results in 55: \[ x - 17 = 55 \] 3. Solve for \( x \): \[ x = 55 + 17 = 72 \] 4. Now, Yoongi wants to divide this number by 9

In [23]:
instruction = ""
cot_trigger = "BOT: "
answer_trigger = "ANSWER: "


# takes a batch of input and completion strings
# returns a list of completion strings
def extract_completion_batch(input_and_completion_batch):
    cot_trigger_count_in_instructions = instruction.count(cot_trigger)
    splitted = [res.split(cot_trigger) for res in input_and_completion_batch]
    return [
        cot_trigger.join(split[cot_trigger_count_in_instructions + 1 :])
        for split in splitted
    ]


# assumes to get back only generated tokens
# returns empty string if no answer trigger is found
def extract_answer_cot_batch(answer_cot_batch):
    splitted_batch = [res.split(answer_trigger) for res in answer_cot_batch]
    return [
        answer_trigger.join(splitted[1:]) if len(splitted) >= 2 else ""
        for splitted in splitted_batch
    ]


def check_answer(extracted_ans, target_answer):
    extracted_ans = extracted_ans.lower()
    target_answer = target_answer.lower()
    if extracted_ans.strip() == target_answer:
        return 1
    if extracted_ans.strip().startswith(target_answer):
        return 0.5
    return 0


In [26]:
cots = extract_completion_batch([text])
print(cots)
# print #tokens of cot
print(len(tokenizer.encode(cots[0], add_special_tokens=False)))
answers = extract_answer_cot_batch(cots)
print(answers)
check_answer(answers[0], example["final_target"].values[0])


['1. Let the number be \\( x \\). 2. According to the problem, subtracting 17 from \\( x \\) results in 55: \\[ x - 17 = 55 \\] 3. Solve for \\( x \\): \\[ x = 55 + 17 = 72 \\] 4. Now, Yoongi wants to divide this number by 9. \\[ \\text{Quotient} = x / 9 \\] 5. Substituting the calculated value of \\( x \\): \\[ \\text{Quotient} = 72 / 9 = 8 \\] 6. The division result is 8. ANSWER: 8<eos>']
141
['8<eos>']


0.5